In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [13]:
df = pd.read_csv('../Data/train-2026.csv')
print(df.shape)
df.head(5)

/var/folders/22/3z4yr_bd5ds4d1g7ph9nprs80000gn/T/ipykernel_69996/3753667672.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../Data/train-2026.csv')


(1023552, 44)


,Unnamed: 0,id,stmt_date,HQ_city,legal_struct,ateco_sector,def_date,fs_year,asst_intang_fixed,asst_tang_fixed,...,inc_extraord,taxes,profit,days_rec,ebitda,roa,roe,wc_net,margin_fin,cf_operations
0,17,520288,2011-12-31,28.0,SRL,14.0,NaN,2011,67537.0,1137566.0,...,-309.0,-80959.0,-81288.0,NaN,6318.0,-3.81,-28.03,496258.0,-917029.0,-849.0
1,18,520288,2008-12-31,28.0,SRL,14.0,NaN,2008,256438.0,1181416.0,...,-678.0,-94622.0,-107382.0,NaN,46088.0,-2.76,NaN,97952.0,NaN,-3881.0
2,19,520288,2009-12-31,28.0,SRL,14.0,NaN,2009,194046.0,1152014.0,...,4224.0,-74235.0,-77819.0,NaN,67611.0,-2.17,NaN,-210671.0,NaN,32618.0
3,21,520288,2012-12-31,28.0,SRL,14.0,NaN,2012,15195.0,1116938.0,...,3634.0,-250786.0,-250786.0,NaN,-161478.0,-12.99,NaN,367892.0,-1094962.0,-168907.0
4,22,520288,2007-12-31,28.0,SRL,14.0,NaN,2007,126603.0,1127807.0,...,820.0,92192.0,36733.0,NaN,153060.0,6.20,52.43,-317007.0,-1184970.0,80039.0


In [14]:
identifier_and_date_cols = ['id', 'stmt_date', 'def_date', 'fs_year']
firm_characteristics_cols = ['HQ_city', 'legal_struct', 'ateco_sector']
asset_cols = ['asst_intang_fixed', 'asst_tang_fixed', 'asst_fixed_fin', 'asst_current', 'AR', 'cash_and_equiv', 'asst_tot', 'goodwill']
equity_capital_cols = ['eqty_tot', 'eqty_corp_family_tot', 'margin_fin']
debt_cols = ['liab_lt', 'liab_lt_emp', 'debt_bank_st', 'debt_bank_lt', 'debt_fin_st', 'debt_fin_lt', 'AP_st', 'AP_lt', 'debt_st', 'debt_lt']
revenue_and_expenses_cols = ['rev_operating', 'COGS', 'inc_financing', 'exp_financing', 'inc_extraord', 'taxes']
profitability_cols = ['prof_operations', 'prof_financing', 'profit', 'ebitda', 'roa', 'roe']
liquidity_and_cashflows_cols = ['wc_net', 'cf_operations']
ops_efficiency_cols = ['days_rec']

In [15]:
column_mapping = {
    'identifier_and_date_cols' : identifier_and_date_cols,
    'firm_characteristics_cols' : firm_characteristics_cols,
    'asset_cols' : asset_cols,
    'equity_capital_cols' : equity_capital_cols,
    'debt_cols' : debt_cols,
    'revenue_and_expenses_cols' : revenue_and_expenses_cols,
    'profitability_cols' : profitability_cols,
    'liquidity_and_cashflows_cols' : liquidity_and_cashflows_cols,
    'ops_efficiency_cols' : ops_efficiency_cols
}

In [16]:
import pickle

pickle.dump(column_mapping, open('../Files/column_mapping.pkl', 'wb'))

In [17]:
data_statistics = {}
for _k, _v in column_mapping.items():
    data_statistics[_k] = {}
    data_statistics[_k]['description'] = df[_v].describe()
    null_counts = pd.DataFrame(df[_v].isna().sum()).reset_index()
    null_counts.columns = ['feature', 'null_count']
    data_statistics[_k]['null_information'] = null_counts

pickle.dump(data_statistics, open('../Files/data_statistics.pkl', 'wb'))

In [18]:
t_all = pd.DataFrame(columns=['feature','null_count'])
for _k in data_statistics.keys():
    t = data_statistics[_k]['null_information'].query('null_count > 0')
    t_all = pd.concat([t_all, t])

In [19]:
from utils import create_target_variable

df['default'] = df.apply(lambda row: create_target_variable(row['stmt_date'], row['def_date']), axis=1)
df['default'].isna().sum(), df.shape[0]
df = df.loc[~df['default'].isna()]

df.groupby(['fs_year']).agg({'id':'count','default':['sum', 'mean']})

id default          
          count     sum      mean
fs_year                          
2007     144128    1475  0.010234
2008     161136    2308  0.014323
2009     170125    2867  0.016852
2010     174229    2967  0.017029
2011     183730    2134  0.011615
2012     184989       0       0.0

In [20]:
df.to_csv('../Data/train-2026_with_target_variable.csv')

In [21]:
df['data_type'] = df['fs_year'].apply(lambda x: 'train' if x <= 2009 else ('oot1' if x <= 2010 else 'oot2'))
df = df.loc[df['fs_year'] != 2012]
df.groupby(['fs_year','data_type']).agg({'data_type':'count','default':'mean'})

,,data_type,default
fs_year,data_type,,
2007,train,144128,0.010234
2008,train,161136,0.014323
2009,train,170125,0.016852
2010,oot1,174229,0.017029
2011,oot2,183730,0.011615


In [22]:
df.to_csv('../Data/temp_training.csv')

In [3]:
df = pd.read_csv('../Data/temp_training.csv', index_col=0)

/var/folders/22/3z4yr_bd5ds4d1g7ph9nprs80000gn/T/ipykernel_22658/1191397029.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../Data/temp_training.csv', index_col=0)


In [45]:
len(list(df.query('default == 1')['id'])), len(list(set(list(df.query('default == 1')['id']))))

(11751, 11751)

In [46]:
firm_ids = list(df.query('default == 1')['id'])

In [49]:
df["default_year"] = pd.to_datetime(df["def_date"], errors="coerce").dt.year
df.query('id in @firm_ids').groupby(['default_year','fs_year']).agg({'id':'count','default':'sum'}).reset_index()

,default_year,fs_year,id,default
0,2008.0,2007,481,481
1,2009.0,2007,1624,994
2,2009.0,2008,782,782
3,2010.0,2007,2170,0
4,2010.0,2008,2581,1526
5,2010.0,2009,1184,1184
6,2011.0,2007,2122,0
7,2011.0,2008,2441,0
8,2011.0,2009,2800,1683
9,2011.0,2010,1275,1275


In [50]:
# Firms that defaulted in 2009 and have a 2007 statement
firms = df.loc[(df["default_year"] == 2009) & (df["fs_year"] == 2007), "id" ].unique()
firms_with_2008 = df.loc[df["fs_year"] == 2008, "id"].unique()

check = pd.DataFrame({"id": firms})
check["has_2008_statement"] = check["id"].isin(firms_with_2008)

display(check)

# Firms missing the 2008 statement
missing_2008 = check.loc[~check["has_2008_statement"]]
display(missing_2008)

print(f"Firms checked: {len(check):,}")
print(f"Missing 2008 statement: {len(missing_2008):,}")

,id,has_2008_statement
0,3880283,False
1,4750048,True
2,20250411,False
3,40720575,False
4,44090298,False
...,...,...
1883,80004670677,False
1884,80018440588,False
1885,80034750010,True
1886,84004190017,False


,id,has_2008_statement
0,3880283,False
2,20250411,False
3,40720575,False
4,44090298,False
6,52870524,False
...,...,...
1878,13456230153,False
1882,80001700261,False
1883,80004670677,False
1884,80018440588,False


Firms checked: 1,888
Missing 2008 statement: 1,258


In [4]:
df.query('default == 1').isna().sum()

Unnamed: 0                  0
id                          0
stmt_date                   0
HQ_city                    13
legal_struct                0
ateco_sector                0
def_date                    0
fs_year                     0
asst_intang_fixed           5
asst_tang_fixed             1
asst_fixed_fin              4
asst_current                0
AR                          0
cash_and_equiv              0
asst_tot                    0
eqty_tot                    0
eqty_corp_family_tot    11751
liab_lt                     5
liab_lt_emp                 5
debt_bank_st             1679
debt_bank_lt             1134
debt_fin_st              1704
debt_fin_lt              1150
AP_st                    1670
AP_lt                    1156
debt_st                     0
debt_lt                     3
rev_operating               2
COGS                        0
prof_operations             0
goodwill                    0
inc_financing               0
exp_financing               4
prof_finan

In [27]:
df = pd.read_csv('../Data/ratio_and_flag_features.csv')

In [33]:
total_cnt = df.shape[0]
default_cnt = df.query('default == 1').shape[0]
total_cnt, default_cnt

(833348, 11751)

In [ ]:

temp2

,features,null_cnts_1
0,id,0
1,data_type,0
2,default,0
3,HQ_city,13
4,ateco_sector,0
5,log_assets,0
6,net_profit_assets,0
7,operating_profit_assets,0
8,operating_margin,638
9,roa,1


In [34]:
temp = pd.DataFrame(df.isna().sum()).reset_index()
temp.columns = ['features', 'null_cnts_all']

temp2 = pd.DataFrame(df.query('default == 1').isna().sum()).reset_index()
temp2.columns = ['features', 'null_cnts_1']

temp = temp.merge(temp2, on=['features'], how='inner')
temp

,features,null_cnts_all,null_cnts_1
0,id,0,0
1,data_type,0,0
2,default,0,0
3,HQ_city,252,13
4,ateco_sector,0,0
5,log_assets,0,0
6,net_profit_assets,20,0
7,operating_profit_assets,21,0
8,operating_margin,28012,638
9,roa,27,1


In [36]:
temp['null_cnts_all_rate'] = temp['null_cnts_all'].apply(lambda x: f"{x / total_cnt:.4%}")
temp['null_cnts_1_rate'] = temp['null_cnts_1'].apply(lambda x: f"{x / default_cnt:.4%}")
temp

,features,null_cnts_all,null_cnts_1,null_cnts_all_rate,null_cnts_1_rate
0,id,0,0,0.0000%,0.0000%
1,data_type,0,0,0.0000%,0.0000%
2,default,0,0,0.0000%,0.0000%
3,HQ_city,252,13,0.0302%,0.1106%
4,ateco_sector,0,0,0.0000%,0.0000%
5,log_assets,0,0,0.0000%,0.0000%
6,net_profit_assets,20,0,0.0024%,0.0000%
7,operating_profit_assets,21,0,0.0025%,0.0000%
8,operating_margin,28012,638,3.3614%,5.4293%
9,roa,27,1,0.0032%,0.0085%
